###Section G

In [0]:
dbutils.widgets.text("catalog", "vp_ms20010501")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.user_entitlements (
    user_email STRING,
    allowed_city STRING,
    can_view_pii BOOLEAN
) USING DELTA;
""")

###G1

In [0]:
spark.sql(f"""
REVOKE ALL PRIVILEGES ON CATALOG {catalog} FROM `account users`;
""")

spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`;
""")

spark.sql(f"""
GRANT USE SCHEMA, SELECT ON SCHEMA {catalog}.gold TO `account users`;
""")

###
G2

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.gold.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE
  WHEN EXISTS (
    SELECT 1 FROM {catalog}.ops.user_entitlements
    WHERE user_email = current_user() AND can_view_pii = TRUE
  )
  THEN phone
  ELSE CONCAT('XXXXXX', RIGHT(phone, 4))
END;
""")

spark.sql(f"""
ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN phone SET MASK {catalog}.gold.mask_phone;
""")

G3

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.gold.station_row_filter(city STRING)
RETURNS BOOLEAN
RETURN EXISTS (
    SELECT 1 FROM {catalog}.ops.user_entitlements ue
    WHERE ue.user_email = current_user()
    AND (ue.allowed_city = 'ALL' OR ue.allowed_city = city)
);
""")

spark.sql(f"""
ALTER TABLE {catalog}.gold.agg_daily_station SET ROW FILTER {catalog}.gold.station_row_filter ON (city);
""")

###G4

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.governance_evidence (
    evidence_name STRING,
    observed_value STRING
) USING DELTA;
""")

# 1. Set entitlement to Jaipur, can_view_pii = FALSE for current user
spark.sql(f"""
MERGE INTO {catalog}.ops.user_entitlements AS target
USING (SELECT current_user() AS user_email, 'Jaipur' AS allowed_city, FALSE AS can_view_pii) AS source
ON target.user_email = source.user_email
WHEN MATCHED THEN UPDATE SET allowed_city = source.allowed_city, can_view_pii = source.can_view_pii
WHEN NOT MATCHED THEN INSERT *;
""")

# 2. Capture evidence rows
spark.sql(f"""
INSERT INTO {catalog}.ops.governance_evidence
SELECT 'ROW_FILTER_ROW_COUNT_JAIPUR' AS evidence_name, CAST(COUNT(*) AS STRING) AS observed_value
FROM {catalog}.gold.agg_daily_station;
""")

spark.sql(f"""
INSERT INTO {catalog}.ops.governance_evidence
SELECT 'MASKED_PHONE_M0001' AS evidence_name, phone AS observed_value
FROM {catalog}.gold.dim_member
WHERE member_id = 'M0001' LIMIT 1;
""")

# 3. Reset entitlement to ALL, FALSE for Run 3 continuation
spark.sql(f"""
UPDATE {catalog}.ops.user_entitlements 
SET allowed_city = 'ALL', can_view_pii = FALSE 
WHERE user_email = current_user();
""")